# Maximum Distinct Elements After Swaps

## Problem Statement

Given:
- Array **A** of length $n$
- Array **B** of length $m$
- Integer **k** (max swaps allowed)

You may swap `A[i]` with `B[j]` for any `i, j`. Perform **at most k swaps**.

**Goal:** Maximize the number of **distinct elements** in A.

---

## Approach: Greedy

**Key insight:** The only useful swaps are:
- Swapping out a **duplicate** in A
- Swapping in an element from B that A doesn't already contain

**Algorithm:**
1. Find all **duplicate slots** in A (elements that appear more than once)
2. Find all **new values** in B (values not present in A)
3. Greedily swap one duplicate for one new value until we run out of swaps, duplicates, or new values

**Why greedy works:** Every valid swap strictly increases the number of distinct elements by 1. There's no benefit to saving swaps for later — take each one the moment it's available.

**Complexity:** $O(n \log n)$ for counting, $O(n)$ for the greedy sweep.

In [ ]:
from collections import Counter


def max_distinct(a, b, k):
    """
    Maximize distinct elements in A after at most k swaps with B.

    Args:
        a: list of integers (the array to optimize)
        b: list of integers (the pool to draw from)
        k: maximum number of swaps

    Returns:
        int: maximum distinct elements achievable in A

    Time:  O(n log n)
    Space: O(n)
    """
    count_a    = Counter(a)
    unique_in_a = set(a)

    # Values in B that would add a new distinct element to A
    new_from_b = [x for x in b if x not in unique_in_a]

    # Duplicate slots in A — we can swap these out
    duplicates = []
    for val, cnt in count_a.items():
        if cnt > 1:
            duplicates.extend([val] * (cnt - 1))  # cnt-1 are "extra"

    # Greedy: each swap converts one duplicate into one new distinct element
    swaps = 0
    for new_val in new_from_b:
        if swaps >= k or not duplicates:
            break
        duplicates.pop()          # remove a duplicate slot from A
        unique_in_a.add(new_val)  # add the new element to A's distinct set
        swaps += 1

    return len(unique_in_a)

## Walkthrough

```
A = [2, 3, 3, 2, 2]   →   unique: {2, 3},  duplicates: [2, 2, 3]  (3 extra slots)
B = [1, 3, 2, 4, 1]   →   new values (not in A): [1, 4, 1]
k = 2
```

Greedy execution:

| Swap # | New value from B | Duplicate removed | Distinct set | Count |
|--------|-----------------|-------------------|--------------|-------|
| start | — | — | {2, 3} | 2 |
| 1 | 1 | 2 (extra) | {1, 2, 3} | 3 |
| 2 | 4 | 2 (extra) | {1, 2, 3, 4} | 4 |
| (k=2 exhausted) | | | | |

Result: **4**

In [ ]:
def max_distinct_verbose(a, b, k):
    count_a     = Counter(a)
    unique_in_a = set(a)
    new_from_b  = [x for x in b if x not in unique_in_a]

    duplicates = []
    for val, cnt in count_a.items():
        if cnt > 1:
            duplicates.extend([val] * (cnt - 1))

    print(f"A:            {a}")
    print(f"B:            {b}")
    print(f"k:            {k}")
    print(f"Unique in A:  {unique_in_a}")
    print(f"Duplicates:   {duplicates} ({len(duplicates)} swappable slots)")
    print(f"New from B:   {new_from_b}")
    print()

    swaps = 0
    for new_val in new_from_b:
        if swaps >= k or not duplicates:
            break
        removed = duplicates.pop()
        unique_in_a.add(new_val)
        swaps += 1
        print(f"  Swap {swaps}: replace duplicate '{removed}' with new '{new_val}' → distinct={unique_in_a}")

    print(f"\nFinal distinct elements: {len(unique_in_a)}")
    return len(unique_in_a)


a = [2, 3, 3, 2, 2]
b = [1, 3, 2, 4, 1]
k = 2
max_distinct_verbose(a, b, k)

In [ ]:
# Test cases
test_cases = [
    ([2,3,3,2,2], [1,3,2,4,1], 2, 4, "Main example"),
    ([1,2,3],     [4,5,6],     1, 4, "No duplicates in A, add 1 new"),
    ([1,1,1,1],   [2,3,4,5],   3, 4, "All duplicates, k=3"),
    ([1,2,3],     [1,2,3],     5, 3, "B has nothing new — no gain"),
    ([1,1,1],     [2,3,4],     0, 1, "k=0 — no swaps allowed"),
    ([1,1,2,2],   [3,4,5,6],   4, 4, "k > swappable slots"),
]

print(f"{'Description':<35} {'A':<20} {'B':<20} {'k':<4} {'Expected':<10} {'Got':<6} {'Pass?'}")
print("-" * 105)
for a, b, k, expected, desc in test_cases:
    result = max_distinct(a, b, k)
    passed = '✓' if result == expected else '✗'
    print(f"{desc:<35} {str(a):<20} {str(b):<20} {k:<4} {expected:<10} {result:<6} {passed}")

---
## Complexity Analysis

| | Complexity | Reason |
|--|------------|--------|
| Time | $O(n)$ | Counter + set ops are linear; loop is linear |
| Space | $O(n)$ | Counter, set, and duplicates list |

## Common Mistakes

1. **Swapping in values already in A** — check `x not in unique_in_a` before adding to `new_from_b`
2. **Forgetting duplicates in B** — B can have repeated values, but we only care if the value is *new to A*
3. **Off-by-one on duplicates** — an element appearing `cnt` times contributes `cnt - 1` swappable slots
4. **Exceeding k** — track swap count and break when `swaps >= k`